# Curation of Final Multimodal Raw Dataset (ADNI 3-Class: CN, MCI, AD)

This notebook curates the multimodal dataset manifest by linking all preprocessed neuroimaging volumes on disk to the clinical diagnosis labels from `ADNIMERGE_31Dec2025.csv`:

### Diagnostic Classes:
- **CN** (Cognitively Normal): Class `0`
- **MCI** (Mild Cognitive Impairment = Early MCI [EMCI] + Late MCI [LMCI]): Class `1`
- **AD** (Alzheimer's Disease): Class `2`

### Disk Inventory:
- **`data/processed/mri_atlas_registered`**: **5,233** MNI-registered T1 sMRI volumes (`*_mni.nii.gz`) and affine matrices (`*mri2atlas.mat`) across 1,212 unique patients.
- **`data/processed/pet_atlas_registered`**: **1,117** MNI-registered, sMRI-coregistered PET volumes (`*_coreg_mni.nii.gz`) across 593 unique patients.

### Manifests Created:
1. **`data/processed/multimodal_raw_dataset_all_scans.csv`** (and default **`multimodal_raw_dataset.csv`**): **5,003** scan sessions matching the 3 classes.
2. **`data/processed/multimodal_raw_dataset_baseline.csv`**: **1,092** baseline patients (1 scan per patient).


In [1]:
from pathlib import Path
import re
from collections import Counter
import pandas as pd
import numpy as np

# Set base paths (works whether run from project root or notebooks/)
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

ADNIMERGE_PATH = PROJECT_ROOT / "data" / "raw" / "ADNIMERGE_31Dec2025.csv"
MRI_DIR = PROJECT_ROOT / "data" / "processed" / "mri_atlas_registered"
PET_DIR = PROJECT_ROOT / "data" / "processed" / "pet_atlas_registered"

OUT_ALL_CSV = PROJECT_ROOT / "data" / "processed" / "multimodal_raw_dataset_all_scans.csv"
OUT_BASE_CSV = PROJECT_ROOT / "data" / "processed" / "multimodal_raw_dataset_baseline.csv"
OUT_DEFAULT_CSV = PROJECT_ROOT / "data" / "processed" / "multimodal_raw_dataset.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"ADNIMERGE:    {ADNIMERGE_PATH.exists()} ({ADNIMERGE_PATH})")
print(f"MRI atlas dir:{MRI_DIR.exists()} ({MRI_DIR})")
print(f"PET atlas dir:{PET_DIR.exists()} ({PET_DIR})")


Project root: /leonardo_scratch/fast/IscrC_ISAAC/raffaele/multimodal-alzheimer-classification
ADNIMERGE:    True (/leonardo_scratch/fast/IscrC_ISAAC/raffaele/multimodal-alzheimer-classification/data/raw/ADNIMERGE_31Dec2025.csv)
MRI atlas dir:True (/leonardo_scratch/fast/IscrC_ISAAC/raffaele/multimodal-alzheimer-classification/data/processed/mri_atlas_registered)
PET atlas dir:True (/leonardo_scratch/fast/IscrC_ISAAC/raffaele/multimodal-alzheimer-classification/data/processed/pet_atlas_registered)


## 1. Index Preprocessed MNI Volumes on Disk

Extract patient ID (`PTID`, e.g. `002_S_0295`) and scan ID (`scan_id`, e.g. `I238627`) from file names.


In [2]:
def extract_tokens(filename: str):
    m_ptid = re.search(r"([0-9]{3}_S_[0-9]{4})", filename)
    m_scan = re.search(r"_(I[0-9]+)(?:_|$)", filename)
    return (m_ptid.group(1) if m_ptid else None, m_scan.group(1) if m_scan else None)

mri_files = sorted(list(MRI_DIR.glob("*_mni.nii*")))
pet_files = sorted(list(PET_DIR.glob("*_mni.nii*")))

mri_dict = {}
for f in mri_files:
    ptid, scan = extract_tokens(f.name)
    if ptid:
        mri_dict.setdefault(ptid, []).append((scan, f))

pet_dict = {}
for f in pet_files:
    ptid, scan = extract_tokens(f.name)
    if ptid:
        pet_dict.setdefault(ptid, []).append((scan, f))

print(f"Total sMRI volumes on disk: {len(mri_files)} across {len(mri_dict)} unique patients")
print(f"Total PET volumes on disk:  {len(pet_files)} across {len(pet_dict)} unique patients")
print(f"Patients with BOTH modalities on disk: {len(set(mri_dict.keys()) & set(pet_dict.keys()))}")


Total sMRI volumes on disk: 5233 across 1212 unique patients
Total PET volumes on disk:  1117 across 593 unique patients
Patients with BOTH modalities on disk: 593


## 2. Load ADNIMERGE Clinical Diagnosis Metadata

We use the baseline diagnosis (`DX_bl`) to categorize subjects into 3 classes:
- `CN` -> `CN` (Class 0)
- `EMCI`, `LMCI` -> `MCI` (Class 1)
- `AD` -> `AD` (Class 2)
- (`SMC` and missing baseline diagnoses are excluded from the 3-class cohort)


In [3]:
adni = pd.read_csv(ADNIMERGE_PATH, low_memory=False)
bl_adni = adni[adni["VISCODE"] == "bl"].drop_duplicates("PTID").set_index("PTID")

dx_map = {
    "CN": "CN",
    "EMCI": "MCI",
    "LMCI": "MCI",
    "AD": "AD"
}

# Inspect DX_bl distribution for all sMRI volumes on disk
mri_dx_counts = Counter()
for f in mri_files:
    ptid, _ = extract_tokens(f.name)
    if ptid in bl_adni.index:
        dx = bl_adni.loc[ptid, "DX_bl"]
        mri_dx_counts[str(dx)] += 1
    else:
        mri_dx_counts["NOT_IN_BL"] += 1

print("Distribution of DX_bl across all 5,233 sMRI scans on disk:")
for k, v in mri_dx_counts.most_common():
    print(f"  {k:12s}: {v:5d} scans")


Distribution of DX_bl across all 5,233 sMRI scans on disk:
  LMCI        :  1861 scans
  CN          :  1485 scans
  EMCI        :   950 scans
  AD          :   707 scans
  SMC         :   180 scans
  NOT_IN_BL   :    50 scans


## 3. Curate Full Scan-Level Manifest (All 5,003 Scans)

Each row corresponds to an individual MRI scan session on disk, linked to:
- Patient demographics & clinical covariates (`AGE`, `PTGENDER`, `PTEDUCAT`, `APOE4`, `MMSE`)
- MNI sMRI file path
- Co-registered MNI PET file path (if patient has PET)
- Target SafeTensors filename: `<patient_id><scan_id>.safetensors`


In [4]:
all_scans_rows = []

for f in mri_files:
    ptid, scan_id = extract_tokens(f.name)
    if not ptid or ptid not in bl_adni.index:
        continue
    
    patient_info = bl_adni.loc[ptid]
    dx = patient_info.get("DX_bl")
    if dx not in dx_map:
        continue  # Skip SMC or unlisted
    
    orig_class = dx_map[dx]
    label = {"CN": 0, "MCI": 1, "AD": 2}[orig_class]
    
    has_mri = True
    has_pet = ptid in pet_dict
    has_both = has_mri and has_pet
    
    pet_scan_id, pet_path = pet_dict[ptid][0] if has_pet else (None, None)
    st_name = f"{ptid}{scan_id}.safetensors"
    
    all_scans_rows.append({
        "PTID": ptid,
        "RID": patient_info.get("RID"),
        "scan_id": scan_id,
        "original_class": orig_class,
        "label": label,
        "AGE": patient_info.get("AGE"),
        "PTGENDER": patient_info.get("PTGENDER"),
        "PTEDUCAT": patient_info.get("PTEDUCAT"),
        "APOE4": patient_info.get("APOE4"),
        "MMSE": patient_info.get("MMSE"),
        "has_mri": has_mri,
        "has_pet": has_pet,
        "has_both": has_both,
        "mri_scan_id": scan_id,
        "pet_scan_id": pet_scan_id,
        "mri_file": f.name,
        "pet_file": pet_path.name if pet_path else "",
        "safetensor_file": st_name,
    })

df_all_scans = pd.DataFrame(all_scans_rows)

print("=" * 60)
print(f"Total curated scan sessions: {len(df_all_scans)}")
print(f"  • Scans with sMRI:               {df_all_scans['has_mri'].sum()}")
print(f"  • Scans with PET:                {df_all_scans['has_pet'].sum()}")
print(f"  • Scans with BOTH sMRI + PET:    {df_all_scans['has_both'].sum()}")
print("=" * 60)
print("\nClass breakdown (All Scans):")
print(df_all_scans["original_class"].value_counts())
print("\nClass breakdown for scans with BOTH sMRI & PET:")
print(df_all_scans[df_all_scans["has_both"]]["original_class"].value_counts())


Total curated scan sessions: 5003
  • Scans with sMRI:               5003
  • Scans with PET:                2564
  • Scans with BOTH sMRI + PET:    2564

Class breakdown (All Scans):
original_class
MCI    2811
CN     1485
AD      707
Name: count, dtype: int64

Class breakdown for scans with BOTH sMRI & PET:
original_class
MCI    1441
CN      848
AD      275
Name: count, dtype: int64


## 4. Curate Baseline Cohort (1,092 Patients)

A 1-scan-per-subject baseline manifest for subject-level cross-validation or baseline-only experiments.


In [5]:
# Take the first chronological scan for each unique patient in the 3-class cohort
df_baseline = df_all_scans.drop_duplicates(subset=["PTID"]).copy().reset_index(drop=True)

print("=" * 60)
print(f"Total baseline cohort patients: {len(df_baseline)}")
print(f"  • Baseline patients with sMRI:     {df_baseline['has_mri'].sum()}")
print(f"  • Baseline patients with PET:      {df_baseline['has_pet'].sum()}")
print(f"  • Baseline patients with BOTH:     {df_baseline['has_both'].sum()}")
print("=" * 60)
print("\nClass breakdown (Baseline Cohort):")
print(df_baseline["original_class"].value_counts())
print("\nClass breakdown for baseline patients with BOTH sMRI & PET:")
print(df_baseline[df_baseline["has_both"]]["original_class"].value_counts())


Total baseline cohort patients: 1092
  • Baseline patients with sMRI:     1092
  • Baseline patients with PET:      546
  • Baseline patients with BOTH:     546

Class breakdown (Baseline Cohort):
original_class
MCI    599
CN     289
AD     204
Name: count, dtype: int64

Class breakdown for baseline patients with BOTH sMRI & PET:
original_class
MCI    304
CN     158
AD      84
Name: count, dtype: int64


## 5. Verification & Disk Integrity Checks

Confirm that every file referenced in the manifests physically exists on disk.


In [6]:
# Check sMRI files
mri_missing = [f for f in df_all_scans["mri_file"] if not (MRI_DIR / f).exists()]
print(f"Missing sMRI files on disk: {len(mri_missing)}")

# Check PET files (for scans with PET)
pet_files_to_check = [f for f in df_all_scans[df_all_scans["has_pet"]]["pet_file"]]
pet_missing = [f for f in pet_files_to_check if not (PET_DIR / f).exists()]
print(f"Missing PET files on disk:  {len(pet_missing)}")

assert len(mri_missing) == 0, "Missing MRI files detected!"
assert len(pet_missing) == 0, "Missing PET files detected!"
print("\n[SUCCESS] 100% of referenced neuroimaging files verified on disk.")


Missing sMRI files on disk: 0
Missing PET files on disk:  0

[SUCCESS] 100% of referenced neuroimaging files verified on disk.


## 6. Export Dataset Manifests

Exports:
- `multimodal_raw_dataset_all_scans.csv`: 5,003 scans across CN, MCI, AD.
- `multimodal_raw_dataset_baseline.csv`: 1,092 baseline patients.
- `multimodal_raw_dataset.csv`: Default manifest for preprocessing pipelines.


In [7]:
df_all_scans.to_csv(OUT_ALL_CSV, index=False)
df_all_scans.to_csv(OUT_DEFAULT_CSV, index=False)
df_baseline.to_csv(OUT_BASE_CSV, index=False)

print(f"Saved All-Scans Manifest ({len(df_all_scans)} rows) to:\n  -> {OUT_ALL_CSV}\n  -> {OUT_DEFAULT_CSV}")
print(f"Saved Baseline Manifest  ({len(df_baseline)} rows) to:\n  -> {OUT_BASE_CSV}")

# Preview
df_all_scans[["PTID", "scan_id", "original_class", "label", "AGE", "MMSE", "has_both", "safetensor_file"]].head(10)


Saved All-Scans Manifest (5003 rows) to:
  -> /leonardo_scratch/fast/IscrC_ISAAC/raffaele/multimodal-alzheimer-classification/data/processed/multimodal_raw_dataset_all_scans.csv
  -> /leonardo_scratch/fast/IscrC_ISAAC/raffaele/multimodal-alzheimer-classification/data/processed/multimodal_raw_dataset.csv
Saved Baseline Manifest  (1092 rows) to:
  -> /leonardo_scratch/fast/IscrC_ISAAC/raffaele/multimodal-alzheimer-classification/data/processed/multimodal_raw_dataset_baseline.csv


,PTID,scan_id,original_class,label,AGE,MMSE,has_both,safetensor_file
0,002_S_0295,I238627,CN,0,84.8,28.0,True,002_S_0295I238627.safetensors
1,002_S_0295,I303066,CN,0,84.8,28.0,True,002_S_0295I303066.safetensors
2,002_S_0413,I115006,CN,0,76.3,29.0,True,002_S_0413I115006.safetensors
3,002_S_0413,I142972,CN,0,76.3,29.0,True,002_S_0413I142972.safetensors
4,002_S_0413,I240812,CN,0,76.3,29.0,True,002_S_0413I240812.safetensors
5,002_S_0413,I30118,CN,0,76.3,29.0,True,002_S_0413I30118.safetensors
6,002_S_0413,I30119,CN,0,76.3,29.0,True,002_S_0413I30119.safetensors
7,002_S_0413,I304793,CN,0,76.3,29.0,True,002_S_0413I304793.safetensors
8,002_S_0413,I371991,CN,0,76.3,29.0,True,002_S_0413I371991.safetensors
9,002_S_0413,I423202,CN,0,76.3,29.0,True,002_S_0413I423202.safetensors


In [9]:
len(df_all_scans)

5003

In [10]:
df_all_scans

,PTID,RID,scan_id,original_class,label,AGE,PTGENDER,PTEDUCAT,APOE4,MMSE,has_mri,has_pet,has_both,mri_scan_id,pet_scan_id,mri_file,pet_file,safetensor_file
0,002_S_0295,295,I238627,CN,0,84.8,Male,18,1.0,28.0,True,True,True,I238627,I239487,002_S_0295_MPRAGE_I238627_mni.nii.gz,002_S_0295_ADNI_Brain_PET_Raw_FDG_I239487_core...,002_S_0295I238627.safetensors
1,002_S_0295,295,I303066,CN,0,84.8,Male,18,1.0,28.0,True,True,True,I303066,I239487,002_S_0295_MPRAGE_I303066_mni.nii.gz,002_S_0295_ADNI_Brain_PET_Raw_FDG_I239487_core...,002_S_0295I303066.safetensors
2,002_S_0413,413,I115006,CN,0,76.3,Female,16,0.0,29.0,True,True,True,I115006,I240813,002_S_0413_MPRAGE_I115006_mni.nii.gz,002_S_0413_ADNI_Brain_PET_Raw_FDG_I240813_core...,002_S_0413I115006.safetensors
3,002_S_0413,413,I142972,CN,0,76.3,Female,16,0.0,29.0,True,True,True,I142972,I240813,002_S_0413_MPRAGE_I142972_mni.nii.gz,002_S_0413_ADNI_Brain_PET_Raw_FDG_I240813_core...,002_S_0413I142972.safetensors
4,002_S_0413,413,I240812,CN,0,76.3,Female,16,0.0,29.0,True,True,True,I240812,I240813,002_S_0413_MPRAGE_I240812_mni.nii.gz,002_S_0413_ADNI_Brain_PET_Raw_FDG_I240813_core...,002_S_0413I240812.safetensors
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4998,941_S_4420,4420,I364600,MCI,1,81.4,Male,18,0.0,27.0,True,True,True,I364600,I296972,941_S_4420_MPRAGE_I364600_mni.nii.gz,941_S_4420_ADNI_Brain_PET_Raw_FDG_I296972_core...,941_S_4420I364600.safetensors
4999,941_S_4764,4764,I307653,MCI,1,82.7,Female,20,0.0,30.0,True,True,True,I307653,I310349,941_S_4764_MPRAGE_I307653_mni.nii.gz,941_S_4764_ADNI_Brain_PET_Raw_FDG_I310349_core...,941_S_4764I307653.safetensors
5000,941_S_4764,4764,I325119,MCI,1,82.7,Female,20,0.0,30.0,True,True,True,I325119,I310349,941_S_4764_MPRAGE_I325119_mni.nii.gz,941_S_4764_ADNI_Brain_PET_Raw_FDG_I310349_core...,941_S_4764I325119.safetensors
5001,941_S_4764,4764,I376064,MCI,1,82.7,Female,20,0.0,30.0,True,True,True,I376064,I310349,941_S_4764_MPRAGE_I376064_mni.nii.gz,941_S_4764_ADNI_Brain_PET_Raw_FDG_I310349_core...,941_S_4764I376064.safetensors
